# 🏙️ StreetSynth: Free-Tier GPU Inference Notebook
### Human-in-the-Loop Generative AI for Inclusive Street Design

**Author:** Akhil Chivukula | **PRN:** 23070126009 | **Division:** AIML A1  
**Repository:** [https://github.com/sirchivvi/StreetSynth](https://github.com/sirchivvi/StreetSynth)  

---
### Instructions:
1. In Google Colab, select **Runtime > Change runtime type**.
2. Choose **T4 GPU** (Free tier) under Hardware accelerator.
3. Run the cells in sequence to perform high-resolution 1024×512 street accessibility synthesis with zero hosting cost.

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the repository and enter directory
!git clone https://github.com/sirchivvi/StreetSynth.git
%cd StreetSynth

# Install dependencies (torch, torchvision, transformers, timm, albumentations)
!pip install -q -r requirements.txt

## 2. Verify GPU Acceleration

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Accelerated Device:", torch.cuda.get_device_name(0))
    print("VRAM Available:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("Running on CPU (Inference latency: ~30-60s per image)")

## 3. Initialize StreetSynth Inference Pipeline
The pipeline orchestrates:
- **Semantic Segmentation:** SegFormer-B5 / DeepLabV3+ (Cityscapes 19 classes)
- **Monocular Depth:** DPT-Large / MiDaS relative inverse depth
- **Placement Engine:** Deterministic rule-based spatial validation (`cv/placement.py`)
- **AccessNet Generator:** 5-channel conditional U-Net (`accessnet_ep020.pt`)

In [ ]:
import os
from pipeline import StreetSynthPipeline

# Locate checkpoint
CHECKPOINT_PATH = "checkpoints/accessnet_ep020.pt"
if not os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint {CHECKPOINT_PATH} not found. Running up to placement engine.")
    CHECKPOINT_PATH = None

print("Loading StreetSynth Pipeline onto device...")
pipeline = StreetSynthPipeline(
    checkpoint_path=CHECKPOINT_PATH,
    lightweight=True  # Skips LaMa for rapid execution; set False if big-lama is present
)
print("Pipeline Ready!")

## 4. Run Inference on a Street Scene
Supported intervention choices: `"crosswalk"`, `"bench"`, `"curb_ramp"`.

In [ ]:
import os
from PIL import Image

# Choose an input image from test_images or provide your own
test_image = "test_images/sample.jpg"
if not os.path.exists(test_image):
    test_image = "outputs/sample_before.png"

intervention = "crosswalk"  # 'crosswalk' | 'bench' | 'curb_ramp'

print(f"Processing {test_image} for intervention: {intervention}...")
result = pipeline.run(
    image_path=test_image,
    intervention_type=intervention,
    output_dir="outputs"
)

if not result["valid"]:
    print(f"❌ Placement Rejected: {result['reason']}")
else:
    print(f"✅ Placement Succeeded: {result['placement']['reason']}")

## 5. Visualize Intermediate Layers & Synthesized Output

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from cv.segmentation import PALETTE

if result.get("valid", False):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    # 1. Original Input
    axes[0, 0].imshow(result["img_np"])
    axes[0, 0].set_title("1. Original Street Scene (RGB)", fontweight="bold")
    axes[0, 0].axis("off")

    # 2. Semantic Segmentation Map
    seg_color = np.array(PALETTE, dtype=np.uint8)[result["seg_map"]]
    axes[0, 1].imshow(seg_color)
    axes[0, 1].set_title("2. Semantic Segmentation (SegFormer-B5 / DeepLabV3+)", fontweight="bold")
    axes[0, 1].axis("off")

    # 3. Monocular Depth Field
    depth_color = cm.plasma(result["depth_norm"])[:, :, :3]
    axes[1, 0].imshow(depth_color)
    axes[1, 0].set_title("3. Continuous Relative Depth (DPT-Large / MiDaS)", fontweight="bold")
    axes[1, 0].axis("off")

    # 4. AccessNet Synthesized Intervention
    axes[1, 1].imshow(result["final_img"])
    axes[1, 1].set_title(f"4. AccessNet Synthesized ({intervention.upper()})", fontweight="bold", color="green")
    axes[1, 1].axis("off")

    plt.suptitle(f"StreetSynth Output — Accessibility Score: {result['indicators']['overall']['score']}/3", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Accessibility Indicators Scorecard

In [ ]:
from cv.indicators import format_indicators
print(format_indicators(result["indicators"]))
print("\nPlacement Details:", result["placement"])